# Stop-sign violation detector — Colab runner

This notebook is dedicated to Yuval's stop-sign module. It checks out the `stop-sign-module` branch, runs the synthetic tests, processes one video with only the stop-sign module enabled, renders the stop zones, and saves the alerts as JSON.

For a meaningful result, use footage where a stop sign and the approaching vehicle are both visible. A generic motorway sample can verify that the pipeline runs, but it cannot validate stop-sign behaviour.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab choose Runtime > Change runtime type > GPU.")

## 1. Fetch the Stop Sign branch

This is intentionally separate from Ariel's `run_on_colab.ipynb`. Re-running the cell updates an existing clean checkout with a fast-forward pull.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Arielevi15/Crime_Traffic_Dedector.git"
BRANCH = "main"
REPO_DIR = "/content/Crime_Traffic_Dedector"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True
    )

os.chdir(REPO_DIR)

# Drop the package from Python's import cache. Without this, `git pull`
# updates the files on disk while the kernel keeps executing the copies
# already in memory -- a fixed bug reproduces identically and the fix
# looks like it failed.
for _name in [n for n in list(sys.modules) if n.startswith("road_crime")]:
    sys.modules.pop(_name, None)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", commit)

## 2. Install dependencies and run the synthetic tests

In [ ]:
%pip install -q rfdetr trackers

In [ ]:
import subprocess
import sys

# Both suites, not just this one: main stays green only if the other
# track's tests pass too (WORKPLAN rule 0.6).
subprocess.run([sys.executable, "-m", "tests.test_stop_sign"], check=True)
subprocess.run([sys.executable, "-m", "tests.test_wrong_way"], check=True)

## 3. Choose a video

Run **one** of the next two input cells. The upload option is simplest for a short clip; Drive is better for a large file.

In [ ]:
# Option A — upload a short clip from your computer.
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No video was uploaded.")
VIDEO = os.path.abspath(next(iter(uploaded)))
print("Video:", VIDEO)

In [ ]:
# Option B — use a file from Google Drive. Skip this cell when using Option A.
from google.colab import drive

drive.mount("/content/drive")
VIDEO = "/content/drive/MyDrive/dashcam/stop_sign_sample.mp4"
if not os.path.isfile(VIDEO):
    raise FileNotFoundError(f"Update VIDEO to an existing Drive file: {VIDEO}")
print("Video:", VIDEO)

## 4. Run only the Stop Sign module

The RF-DETR class table used by this project maps class ID `13` to `stop sign`; ID `11` is `fire hydrant` and must not be accepted as a stop sign.

In [ ]:
from road_crime.pipeline import run
from road_crime.stop_sign_detector import StopSignConfig

OUTPUT = "/content/stop_sign_result.mp4"
ALERTS_JSON = "/content/stop_sign_alerts.json"
MODEL_VARIANT = "nano"
CONFIDENCE = 0.35
LIMIT_FRAMES = None  # Set an integer such as 300 for a quick smoke test.

stop_config = StopSignConfig()
alerts = run(
    video=VIDEO,
    output=OUTPUT,
    variant=MODEL_VARIANT,
    conf=CONFIDENCE,
    limit_frames=LIMIT_FRAMES,
    modules=("stop_sign",),
    stop_sign_config=stop_config,
)

print(f"Finished with {len(alerts)} alert(s).")

In [ ]:
import json
from pprint import pprint

with open(ALERTS_JSON, "w", encoding="utf-8") as handle:
    json.dump(alerts, handle, indent=2, ensure_ascii=False, allow_nan=False)

pprint(alerts)
print("Alerts JSON:", ALERTS_JSON)

## 5. Preview and download the result

In [ ]:
from IPython.display import Video, display

H264_OUTPUT = "/content/stop_sign_result_h264.mp4"
subprocess.run(
    ["ffmpeg", "-loglevel", "error", "-i", OUTPUT, "-vcodec", "libx264", "-y", H264_OUTPUT],
    check=True,
)
display(Video(H264_OUTPUT, embed=True, width=900))

In [ ]:
from google.colab import files

files.download(H264_OUTPUT)
files.download(ALERTS_JSON)

## Reading the output

- Yellow rectangles are the inferred stop zones.
- A red `STOP SIGN` label is drawn only after a tracked vehicle exits a zone without a measured full stop.
- Zero alerts is not automatically success: confirm that RF-DETR detected the sign, that the zone overlaps the vehicle path, and that the complete approach/exit is present in the clip.
- Threshold tuning and real-footage acceptance are still pending; keep this PR in Draft until those checks are complete.